# Instagram two-agent pipeline (Agent1 unchanged) + Tavily evidence + improved Agent2

- Agent1: splits text into **Discriminative Text** and **Residual Text** (kept identical to your previous logic).
- Tavily: searches for **contact/company legitimacy + scam reports** for discriminative samples.
- Agent2: makes final label using **(a) agent1 output + (b) Tavily evidence**.

> **Label convention used here (based on your `instagram_final_results` errors):**
> - **1 = benign / normal profile**
> - **0 = suspicious / potential scam / unverifiable contact**

Set environment variables before running:
- `OPENAI_API_KEY`, `OPENAI_BASE_URL` (optional), `TAVILY_API_KEY`


In [ ]:
import os
import time

import pandas as pd
from openai import OpenAI

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "Input your key")
OPENAI_BASE_URL = os.getenv("OPENAI_API_BASE", os.getenv("OPENAI_BASE_URL", "Input your URL"))

if not OPENAI_API_KEY:
    raise RuntimeError("Missing OPENAI_API_KEY env var")

client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)

def retry_with_backoff(func, *args, retries=10, delay=2, **kwargs):
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            last_err = e
            if attempt < retries:
                time.sleep(delay)
                delay *= 2
            else:
                raise last_err

def response_API(user_prompt: str, model: str = "gpt-4o-mini", temperature: float = 0.0) -> str:
    def api_call():
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": user_prompt}],
            temperature=temperature,
        )
        return resp.choices[0].message.content

    return retry_with_backoff(api_call, retries=8, delay=2)

CLASSIFIER_PROMPT = r"""
You are a strict binary classifier.
Return ONLY one digit: 0 or 1.

You are given:
- text: full Instagram bio

GOAL:
Match the dataset labels. In this dataset, many bios that look like creators/influencers
are still label=0. Therefore, you must NOT output 1 based on weak “creator vibes”.

A) PRIORITY & NORMALIZATION
1) Ignore corrupted emoji/encoding artifacts like "ðŸ", "ğŸ" etc.

B) CORE DECISION: SCORE + THRESHOLD
Compute a score using ONLY explicit cues in the text.
Default output is 0 unless the score reaches the threshold.

Start score = 0.

Add STRONG POSITIVE signals (+3 each):
+3  Mentions a recognized institution/platform/media network explicitly:
    Examples: "AM 640", "iHeartRadio", "HGTV", "NBC", "radio", "TV presenter"
+3  Clearly indicates an official organization/brand/service venue with operations:
    Examples: salon + booking schedule (W-2-F / appointments / books closed),
    business address + official brand handle, store/clinic/studio name + services.
+3  Clear official topic authority brand like "@SCIENCE" + inquiries email.

Add MODERATE POSITIVE signals (+2 each):
+2  Explicit "Business Inquiries:" / "Press:" / "Media:" + contact email
+2  Two or more consistent external handles pointing to a brand ecosystem
    (e.g., multiple linked brand accounts + role like Founder/Owner)

Add WEAK signals (+1 each) [NEVER sufficient alone]:
+1  Has an email
+1  Has "collab", "DM for collabs"
+1  Has "content creator", "influencer", "blogger"
+1  Has TikTok/YouTube mention
+1  Has generic profession label (photographer, stylist, coach) without external verification

Add STRONG NEGATIVE signals (-3 each):
-3  Pure diary / personal vibe only, no operational or institutional cues:
    examples: "visual diary", "my life", random quotes only
-3  Generic lifestyle list only (travel | fashion | beauty | mindset) with no verifiable anchor
-3  Text is mostly hashtags/emojis with no concrete role/entity

Add MODERATE NEGATIVE signals (-2 each):
-2  Overly generic “creator template” bios:
    "lifestyle | travel | fashion | mindset | growth" + DM/email only
-2  Only location + personal descriptors (mom/wife/dog lover) without institution/entity
-2  The text contains only social handles or hashtags, no real-world anchor

C) OUTPUT THRESHOLD
If score >= 4  -> output 1
Else           -> output 0

IMPORTANT SAFETY GUARDRAILS:
- Never output 1 if the ONLY evidence is: email + "collab" + "lifestyle/blogger".
  (This pattern is frequently label=0 in this dataset.)
- If uncertain, output 0 (because false positives dominate your current errors).
- Return ONLY 0 or 1. No explanation.
""".strip()

def judge_from_row(row: pd.Series, sleep_s: float = 0.0) -> int:
    bio_text = row.get("text", "")

    if not isinstance(bio_text, str) or not bio_text.strip():
        return 0

    user_input = f"Instagram Bio:\n{bio_text}"

    resp = response_API(CLASSIFIER_PROMPT + "\n\n" + user_input)
    time.sleep(sleep_s)

    try:
        val = int(resp.strip())
        return 1 if val == 1 else 0
    except Exception:
        return 0

def run_classifier_and_save_csv(input_csv: str, output_csv: str, sleep_s: float = 0.5):
    df = pd.read_csv(input_csv)

    preds = []
    for idx, row in df.iterrows():
        pred = judge_from_row(row)
        preds.append(pred)
        if idx % 10 == 0:
            print(f"Processed {idx} samples")
        time.sleep(sleep_s)

    df["agent2_pred"] = preds
    df.to_csv(output_csv, index=False)
    print("Final CSV saved to:", output_csv)

if __name__ == "__main__":
    input_csv = "/content/instagram_samples500.csv"
    final_out = "/content/instagram_final_results_direct_classification.csv"

    print("Starting direct classification (Ablation: No Agent1, No Tavily)...")
    run_classifier_and_save_csv(input_csv=input_csv, output_csv=final_out)
    print("Pipeline completed!")

Starting direct classification (Ablation: No Agent1, No Tavily)...
Processed 0 samples
Processed 10 samples
Processed 20 samples
Processed 30 samples
Processed 40 samples
Processed 50 samples
Processed 60 samples
Processed 70 samples
Processed 80 samples
Processed 90 samples
Processed 100 samples
Processed 110 samples
Processed 120 samples
Processed 130 samples
Processed 140 samples
Processed 150 samples
Processed 160 samples
Processed 170 samples
Processed 180 samples
Processed 190 samples
Processed 200 samples
Processed 210 samples
Processed 220 samples
Processed 230 samples
Processed 240 samples
Processed 250 samples
Processed 260 samples
Processed 270 samples
Processed 280 samples
Processed 290 samples
Processed 300 samples
Processed 310 samples
Processed 320 samples
Processed 330 samples
Processed 340 samples
Processed 350 samples
Processed 360 samples
Processed 370 samples
Processed 380 samples
Processed 390 samples
Processed 400 samples
Processed 410 samples
Processed 420 sample

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
# Refactored to use a relative path to ensure cross-platform compatibility
df_result = pd.read_csv("/content/instagram_final_results_direct_classification.csv")

y_true = df_result["label"]
y_pred = df_result["agent2_pred"]

# Retain only rows where both ground truth labels and predictions are valid (non-null)
mask = y_true.notna() & y_pred.notna()

y_true_clean = y_true[mask]
y_pred_clean = y_pred[mask]

# Calculate core binary classification metrics
acc = accuracy_score(y_true_clean, y_pred_clean)
p   = precision_score(y_true_clean, y_pred_clean, average="binary")
r   = recall_score(y_true_clean, y_pred_clean, average="binary")
f1  = f1_score(y_true_clean, y_pred_clean, average="binary")

#F1-macro
f1_macro = f1_score(y_true_clean, y_pred_clean, average="macro")

print("Used rows:", mask.sum(), "/", len(df_result))
print("Accuracy:", acc,
      "Precision:", p,
      "Recall:", r,
      "F1:", f1,
      "F1-macro:", f1_macro)



Used rows: 500 / 500
Accuracy: 0.62 Precision: 0.449438202247191 Recall: 0.22099447513812154 F1: 0.2962962962962963 F1-macro: 0.5180111618467782


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
